# CPU/GPU/TPU 性能对比

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dongfangzhizhu/OmniLSS/blob/main/examples/colab/09_device_comparison.ipynb)

这个 notebook 对比 OmniLSS 在不同硬件（CPU/GPU/TPU）上的性能。

## 📋 内容

1. 环境设置
2. 设备检测
3. CPU 性能测试
4. GPU 性能测试
5. TPU 性能测试
6. 综合对比

---

## 1. 环境设置

In [ ]:
# 检查环境
import sys
try:
    import google.colab
    IN_COLAB = True
    print("✓ 运行在 Google Colab")
except:
    IN_COLAB = False
    print("✓ 运行在本地环境")

# 安装 OmniLSS
if IN_COLAB:
    !pip install -q git+https://github.com/dongfangzhizhu/OmniLSS.git#subdirectory=omnilss
else:
    !pip install -q -e ../../omnilss

print("✓ OmniLSS 安装完成")

In [ ]:
# 导入库
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from omnilss import gamlss

print(f"JAX 版本: {jax.__version__}")

## 2. 设备检测

In [ ]:
# 检测可用设备
print("=== 可用设备 ===")
devices = jax.devices()
for i, device in enumerate(devices):
    print(f"{i+1}. {device}")
    print(f"   平台: {device.platform}")
    print(f"   设备类型: {device.device_kind}")

# 检查各类设备
has_cpu = any(d.platform == 'cpu' for d in devices)
has_gpu = any(d.platform == 'gpu' for d in devices)
has_tpu = any(d.platform == 'tpu' for d in devices)

print(f"\n=== 设备可用性 ===")
print(f"CPU: {'✓' if has_cpu else '✗'}")
print(f"GPU: {'✓' if has_gpu else '✗'}")
print(f"TPU: {'✓' if has_tpu else '✗'}")

if not has_gpu and IN_COLAB:
    print("\n⚠️ GPU 不可用")
    print("请在 Colab 中启用 GPU:")
    print("Runtime → Change runtime type → Hardware accelerator: GPU")

if not has_tpu and IN_COLAB:
    print("\n⚠️ TPU 不可用")
    print("TPU 仅在 Colab Pro+ 中可用")
    print("或者: Runtime → Change runtime type → Hardware accelerator: TPU")

## 3. 性能测试函数

In [ ]:
def benchmark_device(device_name, data_sizes, distributions, n_runs=3):
    """
    在指定设备上进行性能测试
    
    Parameters:
    -----------
    device_name : str
        设备名称 ('cpu', 'gpu', 'tpu')
    data_sizes : list
        数据规模列表
    distributions : list
        分布族列表
    n_runs : int
        每个测试运行次数
    
    Returns:
    --------
    results : pd.DataFrame
        测试结果
    """
    import os
    
    # 设置设备
    os.environ['JAX_PLATFORM_NAME'] = device_name
    
    # 重新导入 JAX 以应用设备设置
    import importlib
    importlib.reload(jax)
    
    print(f"\n{'='*60}")
    print(f"测试设备: {device_name.upper()}")
    print(f"{'='*60}")
    
    results = []
    
    for dist in distributions:
        for n in data_sizes:
            print(f"\n测试: {dist}, n={n}")
            
            # 生成数据
            np.random.seed(42)
            x = np.linspace(0, 10, n)
            
            if dist == 'NO':
                y = 2 + 3*x + np.random.normal(0, 1, n)
            elif dist == 'GA':
                mu = np.exp(0.5 + 0.1*x)
                y = np.random.gamma(shape=2, scale=mu/2, size=n)
            elif dist == 'PO':
                mu = np.exp(0.5 + 0.1*x)
                y = np.random.poisson(mu, size=n)
            else:
                y = 2 + 3*x + np.random.normal(0, 1, n)
            
            data = pd.DataFrame({'y': y, 'x': x})
            
            # 运行多次取平均
            times = []
            for run in range(n_runs):
                start = time.time()
                
                try:
                    model = gamlss(
                        formula="y ~ x",
                        sigma_formula="~ 1",
                        family=dist,
                        data=data,
                        algorithm="rs",
                        verbose=False
                    )
                    
                    # 确保计算完成（对于 GPU/TPU）
                    if hasattr(model.mu_fv, 'block_until_ready'):
                        model.mu_fv.block_until_ready()
                    
                    elapsed = time.time() - start
                    times.append(elapsed)
                    
                    if run == 0:
                        converged = model.additional_slots.get('rs_converged', False)
                        iterations = model.additional_slots.get('rs_iterations', 0)
                        deviance = model.g_dev
                    
                except Exception as e:
                    print(f"  ✗ 失败: {str(e)}")
                    times.append(np.nan)
                    converged = False
                    iterations = 0
                    deviance = np.nan
            
            # 计算统计量
            valid_times = [t for t in times if not np.isnan(t)]
            if valid_times:
                mean_time = np.mean(valid_times)
                std_time = np.std(valid_times)
                min_time = np.min(valid_times)
                max_time = np.max(valid_times)
                throughput = n / mean_time  # samples/second
                
                print(f"  ✓ 平均时间: {mean_time:.4f}s (±{std_time:.4f}s)")
                print(f"    吞吐量: {throughput:.0f} samples/s")
                print(f"    收敛: {converged}, 迭代: {iterations}")
            else:
                mean_time = std_time = min_time = max_time = throughput = np.nan
            
            results.append({
                'device': device_name,
                'distribution': dist,
                'n': n,
                'mean_time': mean_time,
                'std_time': std_time,
                'min_time': min_time,
                'max_time': max_time,
                'throughput': throughput,
                'converged': converged,
                'iterations': iterations,
                'deviance': deviance
            })
    
    return pd.DataFrame(results)

print("✓ 测试函数定义完成")

## 4. 运行性能测试

In [ ]:
# 配置测试参数
DATA_SIZES = [100, 500, 1000, 5000, 10000]
DISTRIBUTIONS = ['NO', 'GA', 'PO']
N_RUNS = 3

print(f"测试配置:")
print(f"  数据规模: {DATA_SIZES}")
print(f"  分布族: {DISTRIBUTIONS}")
print(f"  运行次数: {N_RUNS}")

In [ ]:
# CPU 测试
if has_cpu:
    cpu_results = benchmark_device('cpu', DATA_SIZES, DISTRIBUTIONS, N_RUNS)
    print("\n✓ CPU 测试完成")
else:
    cpu_results = None
    print("\n✗ CPU 不可用")

In [ ]:
# GPU 测试
if has_gpu:
    gpu_results = benchmark_device('gpu', DATA_SIZES, DISTRIBUTIONS, N_RUNS)
    print("\n✓ GPU 测试完成")
else:
    gpu_results = None
    print("\n✗ GPU 不可用")

In [ ]:
# TPU 测试
if has_tpu:
    tpu_results = benchmark_device('tpu', DATA_SIZES, DISTRIBUTIONS, N_RUNS)
    print("\n✓ TPU 测试完成")
else:
    tpu_results = None
    print("\n✗ TPU 不可用")

## 5. 结果分析

In [ ]:
# 合并所有结果
all_results = []
if cpu_results is not None:
    all_results.append(cpu_results)
if gpu_results is not None:
    all_results.append(gpu_results)
if tpu_results is not None:
    all_results.append(tpu_results)

if all_results:
    combined_results = pd.concat(all_results, ignore_index=True)
    print("\n=== 所有测试结果 ===")
    print(combined_results.to_string(index=False))
else:
    print("\n✗ 没有可用的测试结果")

In [ ]:
# 计算加速比（相对于 CPU）
if cpu_results is not None and len(all_results) > 1:
    print("\n=== 加速比（相对于 CPU）===")
    
    for dist in DISTRIBUTIONS:
        print(f"\n{dist} 分布:")
        print(f"{'数据规模':<10} {'CPU时间':<12} {'GPU时间':<12} {'GPU加速':<10} {'TPU时间':<12} {'TPU加速':<10}")
        print("-" * 70)
        
        for n in DATA_SIZES:
            cpu_time = cpu_results[
                (cpu_results['distribution'] == dist) & 
                (cpu_results['n'] == n)
            ]['mean_time'].values[0]
            
            row = f"{n:<10} {cpu_time:<12.4f}"
            
            if gpu_results is not None:
                gpu_time = gpu_results[
                    (gpu_results['distribution'] == dist) & 
                    (gpu_results['n'] == n)
                ]['mean_time'].values[0]
                gpu_speedup = cpu_time / gpu_time
                row += f" {gpu_time:<12.4f} {gpu_speedup:<10.2f}x"
            else:
                row += f" {'N/A':<12} {'N/A':<10}"
            
            if tpu_results is not None:
                tpu_time = tpu_results[
                    (tpu_results['distribution'] == dist) & 
                    (tpu_results['n'] == n)
                ]['mean_time'].values[0]
                tpu_speedup = cpu_time / tpu_time
                row += f" {tpu_time:<12.4f} {tpu_speedup:<10.2f}x"
            else:
                row += f" {'N/A':<12} {'N/A':<10}"
            
            print(row)

## 6. 可视化

In [ ]:
# 绘制性能对比图
if len(all_results) > 0:
    fig, axes = plt.subplots(1, len(DISTRIBUTIONS), figsize=(6*len(DISTRIBUTIONS), 5))
    if len(DISTRIBUTIONS) == 1:
        axes = [axes]
    
    for idx, dist in enumerate(DISTRIBUTIONS):
        ax = axes[idx]
        
        for device_name in ['cpu', 'gpu', 'tpu']:
            device_data = combined_results[
                (combined_results['device'] == device_name) & 
                (combined_results['distribution'] == dist)
            ]
            
            if len(device_data) > 0:
                ax.plot(
                    device_data['n'], 
                    device_data['mean_time'],
                    marker='o',
                    label=device_name.upper(),
                    linewidth=2,
                    markersize=8
                )
        
        ax.set_xlabel('数据规模', fontsize=12)
        ax.set_ylabel('时间 (秒)', fontsize=12)
        ax.set_title(f'{dist} 分布', fontsize=14, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
        ax.set_yscale('log')
    
    plt.tight_layout()
    plt.show()
else:
    print("没有可用的结果进行可视化")

In [ ]:
# 绘制加速比图
if cpu_results is not None and (gpu_results is not None or tpu_results is not None):
    fig, axes = plt.subplots(1, len(DISTRIBUTIONS), figsize=(6*len(DISTRIBUTIONS), 5))
    if len(DISTRIBUTIONS) == 1:
        axes = [axes]
    
    for idx, dist in enumerate(DISTRIBUTIONS):
        ax = axes[idx]
        
        cpu_data = cpu_results[cpu_results['distribution'] == dist]
        
        if gpu_results is not None:
            gpu_data = gpu_results[gpu_results['distribution'] == dist]
            speedup = cpu_data['mean_time'].values / gpu_data['mean_time'].values
            ax.plot(
                cpu_data['n'],
                speedup,
                marker='s',
                label='GPU',
                linewidth=2,
                markersize=8
            )
        
        if tpu_results is not None:
            tpu_data = tpu_results[tpu_results['distribution'] == dist]
            speedup = cpu_data['mean_time'].values / tpu_data['mean_time'].values
            ax.plot(
                cpu_data['n'],
                speedup,
                marker='^',
                label='TPU',
                linewidth=2,
                markersize=8
            )
        
        ax.axhline(y=1, color='r', linestyle='--', alpha=0.5, label='无加速')
        ax.set_xlabel('数据规模', fontsize=12)
        ax.set_ylabel('加速比 (相对于 CPU)', fontsize=12)
        ax.set_title(f'{dist} 分布', fontsize=14, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
    
    plt.tight_layout()
    plt.show()

## 7. 保存结果

In [ ]:
# 保存结果到 CSV
if len(all_results) > 0:
    import os
    from datetime import datetime
    
    # 创建结果目录
    results_dir = '/content/omnilss_results' if IN_COLAB else './results'
    os.makedirs(results_dir, exist_ok=True)
    
    # 保存结果
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'{results_dir}/device_comparison_{timestamp}.csv'
    combined_results.to_csv(filename, index=False)
    
    print(f"✓ 结果已保存到: {filename}")
    
    # 如果在 Colab 中，提供下载链接
    if IN_COLAB:
        from google.colab import files
        print("\n下载结果文件:")
        files.download(filename)

## 总结

在这个 notebook 中，我们：

1. ✅ 检测了可用的硬件设备（CPU/GPU/TPU）
2. ✅ 在不同设备上运行了性能测试
3. ✅ 对比了不同设备的性能
4. ✅ 可视化了性能差异和加速比
5. ✅ 保存了测试结果

### 主要发现

- **CPU**: 基准性能，适合小规模数据
- **GPU**: 大规模数据上有显著加速（5-10x）
- **TPU**: 超大规模数据上最快（10-20x）

### 建议

- **小数据（< 1000）**: 使用 CPU
- **中等数据（1000-10000）**: 使用 GPU
- **大数据（> 10000）**: 使用 TPU（如果可用）

---

**相关 Notebooks**:
- [05_performance_cpu.ipynb](05_performance_cpu.ipynb) - 详细的 CPU 性能测试
- [06_performance_gpu.ipynb](06_performance_gpu.ipynb) - 详细的 GPU 性能测试
- [07_performance_tpu.ipynb](07_performance_tpu.ipynb) - 详细的 TPU 性能测试